##Creating a table of GDPPR healthcare use linked to maternity interpreter cohort 

Purpose - to link GDPPR attendance in year preconception with maternity interpreter cohort

Dependencies - this notebook requires CCU063_03-D01, CCU063_03-D02 and CCU063_03-D03 to have saved their output tables in order to run

Authors - Majel McGranahan supported by Lars Murdock

Reviewed - Not reviewed

#0 Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

In [0]:
checks_on = True

# 1 Load Maternity Interpreter Cohort Table

In [0]:

from pyspark.sql.window import Window
w2 = Window.partitionBy("uniqpregid").orderBy(f.col("est_preg_start"))

maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_previouslosseslessthan24weeks')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (
    maternity_interpreter_cohort
    .select('person_id_mother_deid', 'uniqpregid', 'est_preg_start', 'lookback_start')
    # the below two lines are temporary - should be removed once its one record per pregnancy upstream
   # .withColumn("row", f.row_number().over(w2))
   # .filter(f.col("row") == 1).drop("row") 
)




In [0]:
if checks_on:
    count_var(maternity_interpreter_cohort, 'person_id_mother_deid')
    count_var(maternity_interpreter_cohort, 'uniqpregid')
    display(maternity_interpreter_cohort.limit(10))
    tab(maternity_interpreter_cohort, 'est_preg_start')
    tab(maternity_interpreter_cohort, 'lookback_start')

# 2 Load GDPPR healthcare use
Note - maybe worth checking CCU049 code at this stage

In [0]:
from pyspark.sql import functions as F

latest_date = str(spark.table(F'{dbc_old}.gdppr_{db}_archive').select(F.max(F.col('archived_on') )).collect()[0][0])

spark.table(F'{dbc_old}.gdppr_{db}_archive').printSchema()
print(F'Latest batch available: {latest_date}')
print(F'Current batch being read: {tmp_archived_on}')

In [0]:
gdppr = (
    spark.table(F'{dbc_old}.gdppr_{db}_archive')
    .filter(F.col('archived_on')== tmp_archived_on)
    .select('archived_on', 'NHS_NUMBER_DEID', 'PRACTICE', 'DATE', 'RECORD_DATE','CODE','EPISODE_PRESCRIPTION','VALUE1_CONDITION','VALUE2_CONDITION','VALUE1_PRESCRIPTION','VALUE2_PRESCRIPTION')
    .filter(F.col('DATE').isNotNull())
    .filter(F.col('DATE') >= "2018-01-01") # should change this to dynamic (use a param)
    .filter(F.col('DATE') <= "2024-12-31") # should change this to dynamic (use a param)
    .filter(F.col('EPISODE_PRESCRIPTION').isNull())
    .filter(F.col('VALUE1_PRESCRIPTION').isNull())
    .filter(F.col('VALUE2_PRESCRIPTION').isNull())
    .drop(*['archived_on', 'EPISODE_PRESCRIPTION', 'VALUE1_CONDITION', 'VALUE2_CONDITION'])

)    

In [0]:
if checks_on:
    display(gdppr.limit(5))
    print(gdppr.select(F.min(F.col('DATE') )).collect()[0][0])
    print(gdppr.select(F.max(F.col('DATE') )).collect()[0][0])
    print(gdppr.count()) 


#3 Join GDPPR to Maternity interpreter cohort table 

In [0]:
##Attempting left join based on https://www.geeksforgeeks.org/pyspark-join-types-join-two-dataframes/

##check idcount in each table before and after join (idcount in output table will be same as left table idcount - filtered lookup)

# left join on two dataframes 
maternity_interpreter_GDPPR= (gdppr
                              .join( F.broadcast(maternity_interpreter_cohort), gdppr.NHS_NUMBER_DEID == maternity_interpreter_cohort.person_id_mother_deid,  "inner")
                              .filter(F.col('DATE') > F.col('lookback_start'))
                              .filter(F.col('DATE') < F.col('est_preg_start'))
)


Databricks data profile. Run in Databricks to view.

In [0]:
if checks_on:
    display(maternity_interpreter_GDPPR.sort("person_id_mother_deid", "DATE"), limit=10)
    count_var(maternity_interpreter_GDPPR, 'person_id_mother_deid')
    #count_var(maternity_interpreter_GDPPR, 'uniqpregid')
    #maternity_interpreter_GDPPR.printSchema()
    print(maternity_interpreter_GDPPR.select(F.min(F.col('DATE') )).collect()[0][0])
    print(maternity_interpreter_GDPPR.select(F.max(F.col('DATE') )).collect()[0][0])
    print(maternity_interpreter_GDPPR.select(F.min(F.col('lookback_start') )).collect()[0][0])
    print(maternity_interpreter_GDPPR.select(F.max(F.col('est_preg_start') )).collect()[0][0])
    #print(maternity_interpreter_GDPPR.count()) 

In [0]:
outName = f'{proj}_gp_interaction_details2'

# save
maternity_interpreter_GDPPR.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
#maternity_interpreter_GDPPR = spark.table(f'{dbc}.{proj}_gp_interaction_details')

In [0]:
# Some of the snomed codes in GDPPR would not be indicative of a dialogue/interaction with primary care (such as mass text messages or alerts). For the purposes of this project they would need to be excluded. This codes demonstrates that

# Example: 
# 1240781000000106: Severe acute respiratory syndrome coronavirus 2 vaccination invitation short message service text message sent (situation)
# 1109911000000100: Excepted from cervical screening quality indicators - no response to three invitations (finding)
# 1300561000000107: High risk category for developing complication from coronavirus disease 19 caused by severe acute respiratory syndrome coronavirus 2 infection (finding)
# 171152003: Ca cervix screening - not wanted (situation)
# 185731000: Asthma monitoring call first letter (procedure)
# 956951000000104: Pertussis vaccination in pregnancy (procedure) - ##CHECK THIS COMES UP IN NEW PIPELINE
# 1109921000000106: Quality and Outcomes Framework quality indicator-related care invitation (procedure)
# 185732007: Asthma monitoring call second letter (procedure)
# 391156007: Medication review without patient (procedure)
# 185736005: Asthma monitoring call telephone invite (procedure)
# 1300591000000101: Low risk category for developing complication from coronavirus disease 19 caused by severe acute respiratory syndrome coronavirus 2 infection (finding)
# 1300571000000100: Moderate risk category for developing complication from coronavirus disease 19 caused by severe acute respiratory syndrome coronavirus 2 infection (finding)
# 1324721000000108: Severe acute respiratory syndrome coronavirus 2 vaccination dose declined (situation)
# 1090701000000104: National Health Service Diabetes Prevention Programme invitation (procedure)
# 783401000000101: Stop smoking invitation first short message service text message (procedure)
# 783381000000101: Stop smoking invitation short message service text message (procedure)
# 976631000000101: White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census (finding)
# 976691000000100: White: any other White background - England and Wales ethnic category 2011 census (finding)
# 976791000000107: Asian or Asian British: Indian - England and Wales ethnic category 2011 census (finding)
# 976811000000108: Asian or Asian British: Pakistani - England and Wales ethnic category 2011 census (finding)
# 976871000000103: Asian or Asian British: any other Asian background - England and Wales ethnic category 2011 census (finding)
# 976891000000104: Black or African or Caribbean or Black British: African - England and Wales ethnic category 2011 census (finding)
# 18167009: Black African (ethnic group)
# 414481008: Indian (racial group)
# 494131000000105: White British - ethnic category 2001 census (finding)
# 92391000000108: British or mixed British - ethnic category 2001 census (finding)
# 315236000: White British (ethnic group)
# 92411000000108: Other White background - ethnic category 2001 census (finding)
# 110751000000108: Indian or British Indian - ethnic category 2001 census (finding)
# 92461000000105: Pakistani or British Pakistani - ethnic category 2001 census (finding)
# 92531000000104: Ethnic category not stated - 2001 census (finding)
# 92491000000104: African - ethnic category 2001 census (finding)
# 185984009: White - ethnic group (ethnic group)
# 92481000000101: Other Asian background - ethnic category 2001 census (finding)
# 110761000000106: English - ethnic category 2001 census (finding)
# 92471000000103: Bangladeshi or British Bangladeshi - ethnic category 2001 census (finding)
# 92521000000101: Other - ethnic category 2001 census (finding)
# 94041000000106: Other White European or European unspecified or Mixed European - ethnic category 2001 census (finding)
# 92451000000107: Other Mixed background - ethnic category 2001 census (finding)
# 92511000000107: Chinese - ethnic category 2001 census (finding)
# 92421000000102: White and Black Caribbean - ethnic category 2001 census (finding)
# 94051000000109: Other White or White unspecified - ethnic category 2001 census (finding)
# 88941000000100: Polish - ethnic category 2001 census (finding)
# 92431000000100: White and Black African - ethnic category 2001 census (finding)
# 107691000000105: Caribbean - ethnic category 2001 census (finding)
# 94151000000105: Any other group - ethnic category 2001 census (finding)
# 976871000000103: Asian or Asian British: any other Asian background - England and Wales ethnic category 2011 census (finding)
# 976971000000106: Other ethnic group: any other ethnic group - England and Wales ethnic category 2011 census (finding)
# 92441000000109: White and Asian - ethnic category 2001 census (finding)
# 89001000000105: Arab - ethnic category 2001 census (finding)
# 186002003: Pakistani (ethnic group)

maternity_interpreter_GDPPR2 = (maternity_interpreter_GDPPR
       .filter(f.col("CODE") != "1240781000000106")
       .filter(f.col("CODE") != "1109911000000100")
       .filter(f.col("CODE") != "1300561000000107")
       .filter(f.col("CODE") != "171152003")
       .filter(f.col("CODE") != "185731000")
       .filter(f.col("CODE") != "956951000000104")
       .filter(f.col("CODE") != "1109921000000106")
       .filter(f.col("CODE") != "185732007")
       .filter(f.col("CODE") != "391156007")
       .filter(f.col("CODE") != "185736005")
       .filter(f.col("CODE") != "1300591000000101")
       .filter(f.col("CODE") != "1300571000000100")
       .filter(f.col("CODE") != "1324721000000108")
       .filter(f.col("CODE") != "1090701000000104")
       .filter(f.col("CODE") != "783401000000101")
       .filter(f.col("CODE") != "783381000000101")
       .filter(f.col("CODE") != "976631000000101")
       .filter(f.col("CODE") != "976691000000100")
       .filter(f.col("CODE") != "976791000000107")
       .filter(f.col("CODE") != "976811000000108")
       .filter(f.col("CODE") != "976871000000103")
       .filter(f.col("CODE") != "976891000000104")
       .filter(f.col("CODE") != "18167009")
       .filter(f.col("CODE") != "414481008")
       .filter(f.col("CODE") != "494131000000105")
       .filter(f.col("CODE") != "92391000000108")
       .filter(f.col("CODE") != "315236000")
       .filter(f.col("CODE") != "92411000000108")
       .filter(f.col("CODE") != "110751000000108")
       .filter(f.col("CODE") != "92461000000105")
       .filter(f.col("CODE") != "92531000000104")
       .filter(f.col("CODE") != "92491000000104")
       .filter(f.col("CODE") != "185984009")
       .filter(f.col("CODE") != "92481000000101")
       .filter(f.col("CODE") != "110761000000106")
       .filter(f.col("CODE") != "92471000000103")
       .filter(f.col("CODE") != "92521000000101")
       .filter(f.col("CODE") != "94041000000106")
       .filter(f.col("CODE") != "92451000000107")
       .filter(f.col("CODE") != "92511000000107")
       .filter(f.col("CODE") != "92421000000102")
       .filter(f.col("CODE") != "94051000000109")
       .filter(f.col("CODE") != "88941000000100")
       .filter(f.col("CODE") != "92431000000100")
       .filter(f.col("CODE") != "107691000000105")
       .filter(f.col("CODE") != "94151000000105")
       .filter(f.col("CODE") != "976871000000103")
       .filter(f.col("CODE") != "976971000000106")
       .filter(f.col("CODE") != "92441000000109")
       .filter(f.col("CODE") != "89001000000105")
       .filter(f.col("CODE") != "186002003")
       .filter(f.col("CODE").isNotNull())
)

In [0]:
count_var(maternity_interpreter_GDPPR2, 'person_id_mother_deid')



In [0]:
outName = f'{proj}_gp_interaction_details'

# save
maternity_interpreter_GDPPR2.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

#4. Collapse to one row per date for GP attendances

In [0]:
w2 = Window.partitionBy("person_id_mother_deid").orderBy("person_id_mother_deid")

maternity_interpreter_GDPPR0 = (
    maternity_interpreter_GDPPR2
    .select('person_id_mother_deid', 'uniqpregid', 'DATE')
    .dropDuplicates() #in order to separate to one visit per day
    .withColumn("row", f.row_number().over(w2))
    .withColumn("number_of_gp_interaction_days", f.max(f.col("row")).over(w2))
    .filter(f.col("row") == f.col("number_of_gp_interaction_days"))
    .drop("row") 
)

In [0]:
if checks_on:
    #maternity_interpreter_GDPPR0.printSchema()
    count_var( maternity_interpreter_GDPPR0, 'person_id_mother_deid')
    #maternity_interpreter_GDPPR0.count()
    display(maternity_interpreter_GDPPR0.orderBy("number_of_gp_interaction_days", ascending=False).limit(100))
    tab(maternity_interpreter_GDPPR0, "number_of_gp_interaction_days")

In [0]:
tab(maternity_interpreter_GDPPR0, "number_of_gp_interaction_days")

#5. Link cohort back to maternity interpreter cohort 
This is just so we have all the individuals who did not attend GP in year preconception in the table again!

##5b. Load maternity interpreter cohort again

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_previouslosseslessthan24weeks')
from pyspark.sql.window import Window
w2 = Window.partitionBy("uniqpregid").orderBy(f.col("est_preg_start"))

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (
        maternity_interpreter_cohort
        # the below three lines are temporary - should be removed once its one record per pregnancy upstream
       # .withColumn("row", f.row_number().over(w2))
       # .filter(f.col("row") == 1)
       # .drop("row") 
        .select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'person_id_demo', 'Dob', 'eth5', 'region', 'imd_quintile', 'imd_decile', 'in_gdppr', 'gdppr_min_date', 'interpreter_use', 'record_before_lookback', 'ageatbookingmother', 'delivery_date', 'agefinal', 'folicacid', 'ovsvischcat', 'ovsvischcatappdate', 'complexsocialfactors', 'gestagebooking', 'gestagebookingweeks', 'previouslivebirths', 'previousstillbirths', 'previouslosseslessthan24weeks', 'parity', 'nulliparous', 'prev_loss', 'prev_preg', 'booking_after_10weeks')
        )

In [0]:
count_var(maternity_interpreter_cohort, 'person_id_mother_deid')
count_var(maternity_interpreter_cohort, 'uniqpregid')

# 6. Join GDPPR MSDS cohort to original interpreter cohort again


In [0]:
maternity_interpreter_GDPPR1 = (maternity_interpreter_cohort
       .join(maternity_interpreter_GDPPR0, on = ['person_id_mother_deid', 'uniqpregid' ]    , how = 'left' ) 
       .withColumn('number_of_gp_interaction_days', f.when(f.isnull('number_of_gp_interaction_days'), f.lit(0)).otherwise(f.col('number_of_gp_interaction_days')) )
       .withColumn('fact_of_gp_interaction', f.when(f.col('number_of_gp_interaction_days') > 0, f.lit('One_or_more')).otherwise(f.lit('No_interactions')) )
       #.where( f.col('ageatbookingmother') >= 18) # temporary - please apply this filter further upstream in pipeline - also consider upper bound age as well - appears to be a long tail in the data
)

In [0]:
outName = f'{proj}_cohort_gp_interaction_counts'

# save
maternity_interpreter_GDPPR1.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
##check
if checks_on:
    display(maternity_interpreter_GDPPR1.limit(5))
    #count_var(maternity_interpreter_GDPPR1, 'uniqpregid')
    #maternity_interpreter_GDPPR1.printSchema()

In [0]:
#Load table from database
#maternity_interpreter_GDPPR1 = spark.table(f'{dbc}.{proj}_cohort_gp_interaction_counts')

#7. Work out proportion of patients who attended GP in year preconception

In [0]:
tab(maternity_interpreter_GDPPR1, 'fact_of_gp_interaction')
tab(maternity_interpreter_GDPPR1, 'number_of_gp_interaction_days')

In [0]:
tab(maternity_interpreter_GDPPR1, 'fact_of_gp_interaction', 'interpreter_use')

# 8. Covid breakdown - - TO DISCUSS!

## Filter to pre-COVID

In [0]:
maternity_interpreter_GDPPRX = (
    maternity_interpreter_GDPPR1
    .withColumn('COVID_period_applicable', f.when(f.col('est_preg_start') < '2020-03-01', f.lit('Conception_before_Covid_start'))
                .when(f.col('lookback_start') >= '2020-03-01', f.lit('Lookback_period_after_Covid_start'))
                .otherwise(f.lit('Lookback_period_spans_Covid_start')) )
    )

In [0]:
display(maternity_interpreter_GDPPRX.select("COVID_period_applicable", "person_id_mother_deid", "lookback_start", "est_preg_start",  "interpreter_use").limit(100))

In [0]:
tab(maternity_interpreter_GDPPRX, 'COVID_period_applicable')

In [0]:
display(tab(maternity_interpreter_GDPPRX, 'COVID_period_applicable', 'fact_of_gp_interaction'))

##Interpreter users by COVID GP attendance

In [0]:
maternity_interpreter_GDPPRX2 = (maternity_interpreter_GDPPRX
                    .where( f.col('interpreter_use') == "yes")

)

In [0]:
display(tab(maternity_interpreter_GDPPRX2, 'COVID_period_applicable', 'fact_of_gp_interaction'))

## Non interpreter users by COVID GP attendance

In [0]:
maternity_interpreter_GDPPRX3 = (maternity_interpreter_GDPPRX
                    .where( f.col('interpreter_use') == "no")

)

In [0]:
display(tab(maternity_interpreter_GDPPRX3, 'COVID_period_applicable', 'fact_of_gp_interaction'))

#9. Savepoint including COVID period

In [0]:
outName = f'{proj}_cohort_gp_interaction_counts2'

# save
maternity_interpreter_GDPPRX.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')